# Univariant Screening

In [62]:
import pandas as pd 
import numpy as np
from scipy.stats import ttest_ind, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold

import csv

In [ ]:
#Used same code as step 6
SEED = 42
N_REPEATS = 10    

clean = pd.read_csv("clean_table.csv")
sensors = [c for c in clean.columns if c.startswith("Sensor_")]
X = clean[sensors]
y = clean["label"]                     # 1 = fail, 0 = pass

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)   # identical to Denver's notebook

print("sensors:", len(sensors))
print("train:", X_train.shape, "fails:", y_train.sum())
print("test: ", X_test.shape, "fails:", y_test.sum())

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REPEATS, random_state=SEED)


sensors: 446
train: (1253, 446) fails: 83
test:  (314, 446) fails: 21


## 5.1 Statistical Testing

In [ ]:
results = []
alpha = 0.05 #this is for FDR level

for s in X_train:
    pass_val = X_train.loc[y_train == 0, s].dropna()
    fail_val = X_train.loc[y_train == 1, s].dropna()
    #print("Passed", len(pass_val))
    #print("Failed", len(fail_val))
    Welch_t_stat, Welch_p_value = ttest_ind(fail_val, pass_val, equal_var = False)
    Mann_t_stat, Mann_p_value = mannwhitneyu(fail_val, pass_val, alternative='two-sided')
    
    #5.2 
    mean_fail = np.mean(fail_val)
    mean_pass = np.mean(pass_val)
    std_fail = np.std(fail_val, ddof=1)
    std_pass = np.std(pass_val, ddof = 1)
    num_fail = len(fail_val)
    num_pass = len(pass_val)

    pooled_std = np.sqrt(((num_pass - 1) * std_pass**2 + (num_fail -1) * std_fail**2) / (num_pass + num_fail -2))
    if pooled_std > 0: 
        effect_size = (mean_fail - mean_pass) / pooled_std
    else: 
        effect_size = np.nan

    
    results.append({"Sensor": s, "Welch P value": Welch_p_value, "Welch T Statistic:": Welch_t_stat, "Mann P Value": Mann_p_value, "Mann_t_stat": Mann_t_stat, "Effect Size": effect_size})
    
    
resultTable = pd.DataFrame(results)

#5.3
reject, p_adjusted, _, _ =  multipletests(resultTable["Welch P value"], alpha = alpha, method="fdr_bh")

resultTable["Q Value (Welch)"] = p_adjusted
resultTable["Significant (Welch)"] = reject

reject2, p_adjusted2, _, _ =  multipletests(resultTable["Mann P Value"], alpha = alpha, method="fdr_bh")

resultTable["Q Value (Mann)"] = p_adjusted2
resultTable["Significant (Mann)"] = reject2

display(resultTable)

print("Significant Sensors:", resultTable["Significant (Welch)"].sum()) # Result in 18 sensors 

print("Significant Sensors:", resultTable["Significant (Welch)"].sum()) # Result is 18 sensors 


,Sensor,Welch P value,Welch T Statistic:,Mann P Value,Mann_t_stat,Effect Size,Q Value (Welch),Significant (Welch),Q Value (Mann),Significant (Mann)
0,Sensor_001,0.548513,-0.602277,0.074763,42694.0,-0.083789,0.771725,False,0.358543,False
1,Sensor_002,0.442484,0.771146,0.373122,51173.5,0.078696,0.703482,False,0.752997,False
2,Sensor_003,0.437741,0.779414,0.916097,47683.0,0.097042,0.699758,False,0.982162,False
3,Sensor_004,0.735553,-0.338675,0.679145,49319.5,-0.031525,0.896329,False,0.915101,False
4,Sensor_005,0.150922,-1.437233,0.905278,47640.0,-0.043743,0.555759,False,0.982162,False
...,...,...,...,...,...,...,...,...,...,...
441,Sensor_586,0.906442,0.117662,0.550136,50416.0,0.005604,0.953475,False,0.842202,False
442,Sensor_587,0.741880,-0.330322,0.987341,48462.5,-0.035749,0.897169,False,0.993314,False
443,Sensor_588,0.086786,1.730990,0.028430,55488.5,0.207724,0.446468,False,0.226427,False
444,Sensor_589,0.104105,1.641295,0.052051,54696.5,0.191771,0.473784,False,0.296393,False


Significant Sensors: 18
Significant Sensors: 18


## 5.4 CV


In [56]:
counter = pd.Series(0, index = sensors) # count number of significants 
for train_idx, val_idx in cv.split(X_train, y_train):
    Y_fold = y_train.iloc[train_idx]
    X_fold = X_train.iloc[train_idx]
    fold_p_value = []
    

    #only using welch
    for s in sensors: 
        pass_val = X_fold.loc[Y_fold == 0, s].dropna()
        fail_val = X_fold.loc[Y_fold == 1, s].dropna()
        #print("Passed", len(pass_val))
        #print("Failed", len(fail_val))
        Welch_t_stat, Welch_p_value = ttest_ind(fail_val, pass_val, equal_var = False)
        fold_p_value.append(Welch_p_value)

    reject, p_adjusted, _, _ =  multipletests(fold_p_value, alpha = alpha, method="fdr_bh")
    counter = counter + pd.Series(reject.astype(int), index=sensors)
        
        
resultTable["CV Count Significant"] = resultTable["Sensor"].map(counter)
resultTable["CV Frequency"] = resultTable["CV Count Significant"] / 50

#display(resultTable.sort_values("CV Frequency", ascending=False))

In [66]:
significant_sensor_Welch = resultTable[resultTable["Significant (Welch)"] == True]
significant_sensor_sorted = significant_sensor_Welch.sort_values("Q Value (Welch)")
display(significant_sensor_sorted)

significant_sensor_sorted.to_csv("Sorted_Signifcant_Sensors.csv", index = False)

,Sensor,Welch P value,Welch T Statistic:,Mann P Value,Mann_t_stat,Effect Size,Q Value (Welch),Significant (Welch),Q Value (Mann),Significant (Mann),CV Count Significant,CV Count Frequency,CV Frequency
116,Sensor_130,8.167125e-07,5.236375,1.792392e-04,60018.0,0.436446,0.000364,True,0.008882,True,50,1.00,1.00
93,Sensor_104,2.127124e-06,5.070416,3.207784e-08,66057.5,0.702735,0.000474,True,0.000014,True,50,1.00,1.00
112,Sensor_126,8.373150e-05,-4.098934,1.065908e-03,37834.5,-0.381964,0.012448,True,0.032255,True,35,0.70,0.70
54,Sensor_060,1.308139e-04,3.995181,1.138003e-06,63785.5,0.508066,0.014586,True,0.000254,True,31,0.62,0.62
254,Sensor_317,3.131728e-04,-3.734915,8.459591e-04,37890.5,-0.362630,0.021718,True,0.029023,True,19,0.38,0.38
109,Sensor_123,3.408672e-04,-3.707302,2.987964e-03,38793.5,-0.342051,0.021718,True,0.078390,False,20,0.40,0.40
433,Sensor_574,2.721975e-04,-3.736519,5.956946e-02,42553.0,-0.232688,0.021718,True,0.305379,False,22,0.44,0.44
114,Sensor_128,5.627102e-04,-3.559825,7.825621e-03,39772.0,-0.320756,0.031371,True,0.132925,False,15,0.30,0.30
117,Sensor_131,7.213662e-04,3.485992,1.382270e-02,55966.0,0.314031,0.035748,True,0.168802,False,14,0.28,0.28
388,Sensor_511,1.092615e-03,3.379169,1.733392e-05,62137.5,0.582945,0.038596,True,0.002052,True,3,0.06,0.06


In [53]:
#the idea to use Mann doesnt work bc for the CV we only used welch
#significant_sensor_Mann = resultTable[resultTable["Significant (Mann)"] == True]
#display(significant_sensor_Mann.sort_values("Q Value (Mann)"))

In [68]:
condenced_table = significant_sensor_Welch[["Sensor", "Q Value (Welch)", "Effect Size", "CV Count Significant", "CV Frequency"]].sort_values("CV Frequency", ascending=False)
display(condenced_table)

condenced_table.to_csv("CV_Significant_Count_and_Frequency.csv", index = False)

,Sensor,Q Value (Welch),Effect Size,CV Count Significant,CV Frequency
116,Sensor_130,0.000364,0.436446,50,1.00
93,Sensor_104,0.000474,0.702735,50,1.00
112,Sensor_126,0.012448,-0.381964,35,0.70
54,Sensor_060,0.014586,0.508066,31,0.62
433,Sensor_574,0.021718,-0.232688,22,0.44
109,Sensor_123,0.021718,-0.342051,20,0.40
254,Sensor_317,0.021718,-0.362630,19,0.38
114,Sensor_128,0.031371,-0.320756,15,0.30
117,Sensor_131,0.035748,0.314031,14,0.28
328,Sensor_429,0.038596,-0.108627,13,0.26


## 5.5 Shuffled Label Test


In [ ]:
rng = np.random.default_rng(SEED)
shuffled = pd.Series(rng.permutation(y_train.to_numpy()), index = y_train.index)

p_val_shuffled = []

for s in sensors: 

    pass_val = X_train.loc[shuffled == 0, s].dropna()
    fail_val = X_train.loc[shuffled == 1, s].dropna()
    _, shuffled_p = ttest_ind(fail_val, pass_val, equal_var = False)
    p_val_shuffled.append(shuffled_p)

reject_shuffle, p_shuffle, _, _ =  multipletests(p_val_shuffled, alpha = alpha, method="fdr_bh")
        
        
print("Significant sensors with labels shuffled:", reject_shuffle.sum())


Significant sensors with labels shuffled: 3
